In [43]:
%load_ext autoreload
%autoreload 2


import itertools
from math import prod

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from tqdm import tqdm

from assignment_1.modules.models import (
    fit_bias,
    fit_svd,
    fit_time_bias,
    predict_bias,
    predict_svd,
    predict_time_bias,
)
from assignment_1.modules.util import (
    calc_mse,
    clip_timestamps,
    drop_zero_ratings,
    load_csv,
    random_split,
    time_onehot,
)

DATA_DIR = "../data"  # notebook runs from notebooks/

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
TASK_TRAIN_FILE = "tasks_3_4_train.csv"
TASK_TEST_FILE = "task_3_test.csv"
train = load_csv(TASK_TRAIN_FILE, DATA_DIR)
test = load_csv(TASK_TEST_FILE, DATA_DIR)


In [29]:
train.head(4)

,user,item,timestamp,rating
0,u1838647363,i0846907466,1360914337,3
1,u7093796076,i2832259763,1472021070,4
2,u4779464116,i9244553928,1480231213,5
3,u2107634551,i2346931943,1343026318,4


In [30]:
for name, df in [("train", train), ("test", test)]:
    t = pd.to_datetime(df.timestamp, unit="s")
    odd = (t < "2005-01-01") | (t > "2018-12-31")
    print(f"{name}: {odd.sum():,} rows outside 2005–2018 ({odd.mean():.3%})")

train: 1,944 rows outside 2005–2018 (0.136%)
test: 30 rows outside 2005–2018 (0.150%)


In [31]:
tr, val = random_split(train, val_frac=0.25, seed=0)
tr = drop_zero_ratings(tr)
val = drop_zero_ratings(val)

SCALE = (tr.rating.min(), tr.rating.max())

In [32]:
def val_svd_mse(**params):
    model = fit_svd(tr, rating_scale=SCALE, **params)
    return calc_mse(val.rating, predict_svd(model, val.user, val.item))

base = val_svd_mse(n_factors=1, n_epochs=50)

print(f"baseline val svd MSE = {base:.4f}  ->  2% target ≈ {base * 0.98:.4f}")

baseline val svd MSE = 0.7565  ->  2% target ≈ 0.7413


In [33]:
def val_bias_mse(**params):
    model = fit_bias(tr, rating_scale=SCALE, **params)
    return calc_mse(val.rating, predict_svd(model, val.user, val.item))

bias = val_bias_mse(reg_u=1, reg_i=5, n_epochs=25)
print(f"bias val MSE = {bias:.4f}  ({(1 - bias / base) * 100:+.1f}% vs baseline)")


bias val MSE = 0.7457  (+1.4% vs baseline)


In [34]:
grid = {"reg_u": [0.3, 1, 3, 5], "reg_i": [2, 5, 10, 20], "n_epochs": [25]}

rows = []
# for values in tqdm(itertools.product(*grid.values()), total=prod(map(len, grid.values()))):
#     params = dict(zip(grid, values))
#     s = val_bias_mse(**params)
#     rows.append({**params, "val_mse": s, "vs_base_%": (1 - s / base) * 100})

# results = pd.DataFrame(rows).sort_values("val_mse")
# results.head(10)

In [35]:
X_tr = time_onehot(tr["timestamp"])
X_val = time_onehot(val["timestamp"], columns=X_tr.columns)
assert list(X_val.columns) == list(X_tr.columns)

In [36]:
bias_model = fit_bias(tr, reg_u=3, reg_i=5, n_epochs=25, rating_scale=SCALE)
p_val = predict_bias(bias_model, val.user, val.item)
print(f"bias val MSE: {calc_mse(val.rating, p_val):.4f}   (expect 0.7450)")

resid = val.rating.to_numpy() - p_val
t = pd.to_datetime(clip_timestamps(val["timestamp"]), unit="s")
for name, key in [("weekday", t.dt.weekday), ("month", t.dt.month), ("year", t.dt.year)]:
    print(f"\n{name}:")
    print(pd.Series(resid).groupby(key.to_numpy()).agg(["mean", "size"]).round(3).T)

bias val MSE: 0.7449   (expect 0.7450)

weekday:
              0          1          2  ...          4          5          6
mean     -0.007     -0.001     -0.004  ...     -0.001      0.003      0.003
size  50134.000  50127.000  49810.000  ...  47904.000  49969.000  54452.000

[2 rows x 7 columns]

month:
             1          2          3   ...         10         11         12
mean      0.014      0.003     -0.002  ...     -0.002      0.005     -0.002
size  36650.000  27886.000  30298.000  ...  26697.000  22195.000  24901.000

[2 rows x 12 columns]

year:
         2005     2006     2007  ...       2016       2017    2018
mean    0.215    0.114    0.115  ...     -0.035     -0.053  -0.173
size  620.000  296.000  798.000  ...  59786.000  42626.000  12.000

[2 rows x 14 columns]


/tmp/ipykernel_31126/1912084692.py:7: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  for name, key in [("weekday", t.dt.weekday), ("month", t.dt.month), ("year", t.dt.year)]:


In [37]:
p_tr = predict_bias(bias_model, tr.user, tr.item)
corr = Ridge(alpha=1.0).fit(X_tr, tr.rating.to_numpy() - p_tr)

p_val2 = (p_val + corr.predict(X_val)).clip(*SCALE)
print(f"bias + time correction: {calc_mse(val.rating, p_val2):.4f}")

bias + time correction: 0.7437


In [38]:
fit = fit_time_bias(tr, reg_u=3, reg_i=5, reg_ib=1e9, n_bins=20)    # huge reg_ib -> plain bias model
p = predict_time_bias(fit, val.user, val.item, val["timestamp"])
print(calc_mse(val.rating, p))       # should be close to 0.745

0.7449232546197531


In [42]:
grid = {
    "reg_u":  [1, 3, 5],
    "reg_i":  [3, 5, 10],
    "n_bins": [5, 8, 10],
    "reg_ib": [30, 50, 100],
}

rows = []
for values in tqdm(itertools.product(*grid.values()), total=prod(map(len, grid.values()))):
    params = dict(zip(grid, values))
    fit = fit_time_bias(tr, **params)
    mse = calc_mse(val.rating, predict_time_bias(fit, val.user, val.item, val["timestamp"]))
    rows.append({**params, "mse": mse, "vs_plain_bias": mse - 0.7449})

results = pd.DataFrame(rows).sort_values("mse")
results.head(10)

  0%|          | 0/81 [00:00<?, ?it/s]

100%|██████████| 81/81 [01:00<00:00,  1.35it/s]


,reg_u,reg_i,n_bins,reg_ib,mse,vs_plain_bias
43,3,5,10,50,0.740598,-0.004302
42,3,5,10,30,0.740698,-0.004202
34,3,3,10,50,0.740704,-0.004196
40,3,5,8,50,0.740752,-0.004148
33,3,3,10,30,0.740805,-0.004095
37,3,5,5,50,0.740813,-0.004087
39,3,5,8,30,0.740824,-0.004076
36,3,5,5,30,0.740839,-0.004061
31,3,3,8,50,0.740866,-0.004034
28,3,3,5,50,0.740927,-0.003973


In [44]:
day = lambda df: (df["timestamp"] // 86400).astype("int64")
tr_ud = set(zip(tr["user"], day(tr)))
shared = np.array([(u, d) in tr_ud for u, d in zip(val["user"], day(val))])
print(f"validation rows sharing a (user, day) with training: {shared.mean():.1%}")

# do those days run high/low relative to the model? (residual spread by user-day)
fit = fit_time_bias(tr, reg_u=3, reg_i=5, n_bins=10, reg_ib=50)
resid_tr = tr.rating.to_numpy() - predict_time_bias(fit, tr.user, tr.item, tr["timestamp"])
ud_mean = pd.Series(resid_tr).groupby([tr["user"].to_numpy(), day(tr).to_numpy()]).mean()
print(f"std of per-(user, day) mean residual: {ud_mean.std():.3f}")

validation rows sharing a (user, day) with training: 8.4%
std of per-(user, day) mean residual: 0.808


In [45]:
key_tr = pd.MultiIndex.from_arrays([tr["user"], day(tr)])
ud = pd.Series(resid_tr, index=key_tr).groupby(level=[0, 1]).agg(["sum", "size"])

p_val = predict_time_bias(fit, val.user, val.item, val["timestamp"])
resid_val = val.rating.to_numpy() - p_val
key_val = pd.MultiIndex.from_arrays([val["user"], day(val)])
s = ud.reindex(key_val)                                    # NaN where no same-day training rows

for lam in [0, 1, 2, 5, 10]:
    corr = (s["sum"] / (s["size"] + lam)).fillna(0).to_numpy()   # shrunk same-day mean residual
    print(f"lam={lam:<3} MSE {calc_mse(val.rating, (p_val + corr).clip(1, 5)):.4f}   (now {calc_mse(val.rating, p_val):.4f})")

lam=0   MSE 0.7706   (now 0.7406)
lam=1   MSE 0.7458   (now 0.7406)
lam=2   MSE 0.7413   (now 0.7406)
lam=5   MSE 0.7392   (now 0.7406)
lam=10  MSE 0.7393   (now 0.7406)
